# Trisha — Mid-Size Models
RTX 3050 6GB | batch_size=32 | accumulation_steps=1

In [ ]:
import sys
sys.path.insert(0, "../..")

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.data.dataset import get_dataloaders
from modules.models.factory import TrashClassifier
from modules.training.train import fit

set_seed(SEED)
print(f"Device: {DEVICE}")

In [ ]:
MODELS = [
    "resnet18",
    "resnet34",
    "densenet121",
    "efficientnet_b1",
    "efficientnet_b2",
]

BATCH_SIZE = 32
ACCUMULATION_STEPS = 1
NUM_WORKERS = 4
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 20
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 10

In [ ]:
train_loader, val_loader, val_ds = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

In [ ]:
results = []
for model_name in MODELS:
    model = TrashClassifier(model_name, num_classes=NUM_CLASSES).to(DEVICE)
    res = fit(
        model, train_loader, val_loader,
        name=f"trisha_{model_name}",
        encoder_name=model_name,
        accumulation_steps=ACCUMULATION_STEPS,
        epochs_head=EPOCHS_HEAD,
        epochs_finetune=EPOCHS_FINETUNE,
        lr_head=LR_HEAD,
        lr_finetune=LR_FINETUNE,
        patience=PATIENCE,
        class_weights=val_ds.class_weights,
        device=DEVICE,
    )
    results.append(res)
    print(f">>> {model_name}: val_f1_macro = {res['best_val_f1']:.4f}")

In [ ]:
import pandas as pd
summary = pd.DataFrame(results)
summary = summary.sort_values("best_val_f1", ascending=False)
summary[["name", "best_val_f1", "f1_per_class"]].to_csv(RESULTS / "trisha_summary.csv", index=False)
summary[["name", "best_val_f1", "f1_per_class"]]